<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  6. Conversations without Tools
</h2>

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.1 Differences between Conversations and Agents
</h3>

[**Conversations**](https://docs.mistral.ai/studio-api/agents/agents-api#conversations) improve Chat Completions on different aspects:
- ✅ They have a **persistent state** by default (except if you pass `store=False`)
- ✅ They provide built-in tools (e.g. web search) in addition to functions calling
- ❌ However, they don't work with local models



When you start a new **Conversation**, you have to choose whether:
- You configure all parameters such as `model`, `instructions` and `tools` in the **Conversation**
- Or you configure these parameters in an [**Agent**](https://docs.mistral.ai/studio-api/agents/agents-api#agents) which can be linked to a new conversation via the agent ID

As a result:
- Your **Conversation** must use a `model` OR an `agent_id` parameter. Setting both returns an error.
- In case your **Conversation** is configured via an **Agent**, adding the `instructions` or `tools` parameters in the **Conversation** would return an error

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">Key Takeaway:</strong>
  The Agent is an optional piece of re-usable configuration
</div>

But why use an **Agent** if **stand-alone Conversations** have nearly the same capabilities?

| Capability | With Agent | Without Agent |
| --- | --- | --- |
| Reusable configuration | ✅ | ❌ |
| Updatable configuration during a conversation | ✅ | ❌ |
| Multi-agents capabilities (handoffs) | ✅ | ❌ |


As Conversations are a pre-requisite for using Agents, we'll start with the former in this notebook.

In [ ]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]
mistral = Mistral(api_key=api_key)

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.2 Start a Conversation
</h3>

Unlike chat completions, **conversations are stateful**! When you **start** a conversation:
- A `ModelConversation` object is created to represent the conversation and its metadata. It is identified by a `conversation_id`.
- It returns a `ConversationResponse` object containing the model response and holds a reference to the `conversation_id` so both documents can be linked together.

In [ ]:
response = mistral.beta.conversations.start(
    model="mistral-small-latest",                      # Alternatively, you can use the agent_id parameter and pass an agent ID
    instructions="You are a Senior Python Developer",  # Instructions are similar to a system prompt
    inputs="How can I reverse a Python list?"          # Alternatively, you can provide a list of (dict) messages [{"role": "user", "content": "How to reverse a Python list?"}]
)

In [ ]:
# You receive a ConversationResponse object in return
response

In [ ]:
# The model response is in the 'outputs' list (similar to 'choices' in chat completions)
# The assistant message is represented by an instance of 'MessageOutputEntry'
response.outputs

In [ ]:
# let's check all conversations started so far - There should be a single one if you haven't used this functionality before
# You can see that the id corresponds to the conversation_id from the ConversationResponse object
mistral.beta.conversations.list()

In [ ]:
# Alternatively, you can also query a specific conversation.
# Notice that ModelConversation objects don't store the conversation history, only configuration artifacts.
conversation = mistral.beta.conversations.get(conversation_id=response.conversation_id)
conversation

In [ ]:
# You can query all messages for our conversation: it includes both the input (from user) and output (from assistant) responses
all_messages = mistral.beta.conversations.get_messages(conversation_id=response.conversation_id)
all_messages

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.3 Continue an Existing Conversation
</h3>

You can continue an existing conversation via the **append** method, passing the conversation ID to it so that the correct conversation is retrieved.

In [ ]:
response = mistral.beta.conversations.append(
    conversation_id=response.conversation_id,
    inputs="And how to sort it?",
)

# It also returns a ConversationResponse object
response

In [ ]:
from IPython.display import Markdown
Markdown(response.outputs[0].content)

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.4 Query Entries & Messages
</h3>

- `Messages` are the **MessageOutputEntry** objects returned by the model, as well as the **MessageInputEntry** objects sent by the user.
- `Entries` are broader: they include not only messages but also ToolExecutionEntry (for pre-built tools), FunctionCallEntry (similar to tool calls) and FunctionResultEntry (similar to ToolMessage)

In [ ]:
all_messages = mistral.beta.conversations.get_messages(conversation_id=response.conversation_id)
all_messages

In [ ]:
all_messages.messages

In [ ]:
all_entries = mistral.beta.conversations.get_history(conversation_id=response.conversation_id)
all_entries

In [ ]:
all_entries.entries

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.5 Restart a Conversation from a Specific Entry
</h3>

Now that you know what an entry is, you can restart a conversation from a specific one (you just need its ID).
It's only possible if you have already used the append method.

In [ ]:
new_response = mistral.beta.conversations.restart(
    conversation_id=response.conversation_id,
    from_entry_id=all_entries.entries[1].id,
    inputs="Forget about it, I need to delete it actually"
)

new_response

In [ ]:
# However, we can see that it generated a one-off response which does not modify the previous conversation
mistral.beta.conversations.get_messages(conversation_id=response.conversation_id).messages

In [ ]:
# You need to append the messages on your own and finish the messages list with an user message (or tool message)
msg_to_append = [
    {"role": "user", "content": "Forget about it, I need to delete it actually"}, # Alternatively, MessageInputEntry(role="user", content="...")
    {"role": "assistant", "content": new_response.outputs[0].content},            # Alternatively, MessageOutputEntry(role="assistant", content="...")
    {"role": "user", "content": "Thanks"}
]

test = mistral.beta.conversations.append(
    conversation_id=response.conversation_id,
    inputs=msg_to_append
)

mistral.beta.conversations.get_messages(conversation_id=response.conversation_id).messages

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  6.8 Delete All Conversations
</h3>

In [ ]:
for conversation in mistral.beta.conversations.list():
    id = conversation.id
    mistral.beta.conversations.delete(conversation_id=id)

In [ ]:
mistral.beta.conversations.list()